<a href="https://colab.research.google.com/github/syedbeeban/IE/blob/main/01_All_Optimization_Methods_for_Integral_Equations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Solving integral equations often involves transforming the continuous mathematical problem into a discrete optimization problem. By parameterizing the unknown function—whether through polynomials, Fourier series, or neural networks—we can minimize the "residual" (the error) to find the solution.

To make the following methods clear and comparable, we will use the same **linear Fredholm integral equation of the second kind** for all examples:

$$u(x) - \frac{1}{2} \int_0^1 xt \cdot u(t) dt = \frac{5}{6}x$$

The exact analytical solution to this equation is **$u(x) = x$**. We will cast this equation into an optimization problem: minimizing the squared difference between the left-hand side and the right-hand side.

---

### 1. Local Gradient-Based Optimization (e.g., BFGS)

**How it works:** We assume the solution $u(x)$ can be approximated by a linear combination of basis functions, such as a polynomial $u(x) \approx c_0 + c_1x + c_2x^2$. We substitute this into the integral equation to calculate the error (residual) at various points $x$. We then use a gradient-based optimization algorithm, like BFGS (Broyden–Fletcher–Goldfarb–Shanno), to find the coefficients ($c_0, c_1, c_2$) that minimize this error.

**Best for:** Smooth functions where a good initial guess is available, and the search space is relatively simple.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1. Setup Domain and Integration Grid
x_points = np.linspace(0, 1, 20)  # Collocation points to check the error
t_points = np.linspace(0, 1, 50)  # Points for numerical integration
dt = t_points[1] - t_points[0]

# 2. Define the Residual Function (Objective to minimize)
def objective(c):
    """
    Calculates the sum of squared residuals for the integral equation.
    c: array of polynomial coefficients [c0, c1, c2] for u(x) = c0 + c1*x + c2*x^2
    """
    total_error = 0

    # Precompute u(t) for the integral since it relies on the same coefficients
    u_t = c[0] + c[1]*t_points + c[2]*t_points**2

    for x in x_points:
        # Calculate u(x)
        u_x = c[0] + c[1]*x + c[2]*x**2

        # Calculate the integral: 0.5 * integral( x * t * u(t) dt )
        # Using a simple Riemann sum for numerical integration
        integral_val = 0.5 * np.sum(x * t_points * u_t * dt)

        # Left Hand Side (LHS) and Right Hand Side (RHS)
        lhs = u_x - integral_val
        rhs = (5.0 / 6.0) * x

        # Add squared error at this point x
        total_error += (lhs - rhs)**2

    return total_error

# 3. Run the Optimization
initial_guess = [0.0, 0.0, 0.0]
result = minimize(objective, initial_guess, method='BFGS')

print("Gradient-Based (BFGS) Results:")
print(f"Optimized Coefficients (c0, c1, c2): {result.x}")
print(f"Expected: ~[0, 1, 0] because exact solution is u(x) = x")

---

### 2. Global Heuristic Optimization (e.g., Differential Evolution)

**How it works:** Gradient-based methods can get stuck in local minima if the integral equation is highly nonlinear. Global optimization methods, like Differential Evolution or Genetic Algorithms, use a population of candidate solutions that mutate and combine over generations. They do not require gradients, making them highly robust.

**Best for:** Highly nonlinear integral equations, rough/noisy functions, or when you have no reliable initial guess.

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution

# We will reuse the same domain, integration grid, and objective function logic
x_points = np.linspace(0, 1, 20)
t_points = np.linspace(0, 1, 50)
dt = t_points[1] - t_points[0]

def objective_de(c):
    total_error = 0
    u_t = c[0] + c[1]*t_points + c[2]*t_points**2

    for x in x_points:
        u_x = c[0] + c[1]*x + c[2]*x**2
        integral_val = 0.5 * np.sum(x * t_points * u_t * dt)
        lhs = u_x - integral_val
        rhs = (5.0 / 6.0) * x
        total_error += (lhs - rhs)**2

    return total_error

# 1. Define parameter boundaries instead of an initial guess
# Let's search for coefficients between -5 and 5
bounds = [(-5, 5), (-5, 5), (-5, 5)]

# 2. Run Differential Evolution
result_de = differential_evolution(objective_de, bounds, strategy='best1bin', seed=42)

print("Global Optimization (Differential Evolution) Results:")
print(f"Optimized Coefficients (c0, c1, c2): {result_de.x}")
print(f"Expected: ~[0, 1, 0]")

---

### 3. Machine Learning / Neural Network Optimization (PINNs)

**How it works:** Physics-Informed Neural Networks (PINNs) parameterize the solution $u(x)$ as a deep neural network rather than a fixed polynomial. The optimization process (training) utilizes backpropagation and stochastic gradient descent variants (like Adam) to minimize the integral equation's residual loss function.

**Best for:** High-dimensional integral equations, integro-differential equations, and complex, non-smooth domains where traditional basis functions struggle.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Define the Neural Network Architecture
class IntegralNet(nn.Module):
    def __init__(self):
        super().__init__()
        # A simple network with one hidden layer
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

model = IntegralNet()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 2. Setup Tensors for Domain and Integration
# requires_grad=True is optional here since we are solving an integral equation,
# not a differential equation (no explicit derivatives of u(x) needed).
x_tensor = torch.linspace(0, 1, 20).view(-1, 1)
t_tensor = torch.linspace(0, 1, 50).view(-1, 1)
dt = t_tensor[1] - t_tensor[0]

# 3. Training Loop (Optimization)
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()

    # Evaluate network at collocation points x and integration points t
    u_x = model(x_tensor)
    u_t = model(t_tensor)

    # Vectorized integration:
    # K(x,t) = x * t. We create a 2D matrix of K for all x and t combinations.
    K_matrix = x_tensor @ t_tensor.T

    # Calculate integral using matrix multiplication / broadcasting
    # Sum over the 't' dimension (dim=1)
    integral_val = 0.5 * torch.sum(K_matrix * u_t.T * dt, dim=1).view(-1, 1)

    # Calculate Residual Loss
    lhs = u_x - integral_val
    rhs = (5.0 / 6.0) * x_tensor

    loss = torch.mean((lhs - rhs)**2)

    # Backpropagation
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

# 4. Test the trained model
print("\nNeural Network (Adam) Results:")
test_x = torch.tensor([[0.5], [1.0]])
predicted_u = model(test_x).detach().numpy()
print(f"Prediction at x=0.5: {predicted_u[0][0]:.4f} (Expected: 0.5)")
print(f"Prediction at x=1.0: {predicted_u[1][0]:.4f} (Expected: 1.0)")

---

### Summary of the Approaches

| Method | Approach | Pros | Cons |
| --- | --- | --- | --- |
| **Gradient-Based (BFGS)** | Parametric basis functions minimized via local derivatives. | Very fast, highly accurate for smooth solutions. | Can get trapped in local minima; requires a good initial guess. |
| **Global Heuristic (Diff. Evolution)** | Population-based evolutionary search. | Robust, handles non-linearities well, no initial guess needed. | Slower to converge; computationally heavier per iteration. |
| **Neural Networks (Adam)** | Deep learning approximation of $u(x)$. | Scales well to complex, high-dimensional integro-differential equations. | Can be overkill for simple 1D problems; requires tuning hyperparameters (epochs, learning rate). |

Would you like me to walk through how to adapt any of these scripts to handle a nonlinear integral equation (like a Volterra equation) instead?